# Employee Salary Prediction — Code Explanation

This notebook explains the project workflow and lets you run the cleaning, exploration, visualization, and model evaluation steps. Run it from the repository root or use the path setup below.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    # Also support opening the notebook while the working directory is notebooks/.
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))
RAW_PATH = PROJECT_ROOT / "data" / "raw" / "Messy_Employee_dataset.csv"
CLEAN_PATH = PROJECT_ROOT / "data" / "processed" / "Cleaned_data.csv"

print("Project root:", PROJECT_ROOT)
print("Raw dataset exists:", RAW_PATH.exists())


## 1. Load and inspect the raw dataset

The raw file is kept in `data/raw/`. It is excluded from Git because it contains employee-style identifiers and contact fields.

In [ ]:
if not RAW_PATH.exists():
    raise FileNotFoundError(
        f"Copy Messy_Employee_dataset.csv into: {RAW_PATH}"
    )

raw_df = pd.read_csv(RAW_PATH)
print("Shape:", raw_df.shape)
display(raw_df.head())
display(raw_df.dtypes)


## 2. Data quality checks

Check missing values, duplicates, and the distribution of key categorical columns before cleaning.

In [ ]:
missing_summary = raw_df.isna().sum().sort_values(ascending=False)
display(missing_summary.to_frame("missing_values"))
print("Duplicate rows:", raw_df.duplicated().sum())

for column in ["Performance_Score", "Status", "Remote_Work"]:
    print(f"\\n{column} value counts:")
    display(raw_df[column].value_counts(dropna=False))


## 3. Run the cleaning script

`Cleaning.py` uses paths relative to the project directory. It removes direct identifiers, converts dates and numeric fields, drops rows with missing target salary, separates Department and Region, and saves the processed data. Missing Age values are retained for later training-only imputation.

In [ ]:
from Cleaning import clean_employee_data

clean_df = clean_employee_data(RAW_PATH, CLEAN_PATH)
display(clean_df.head())
print("Cleaned shape:", clean_df.shape)


## 4. Explore salary patterns

These charts are descriptive, not proof of causation. They help identify distributions and group-level differences in this dataset.

In [ ]:
sns.set_theme(style="whitegrid")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(data=clean_df, x="Salary", kde=True, ax=axes[0])
axes[0].set_title("Salary distribution")

sns.boxplot(data=clean_df, x="Department", y="Salary", ax=axes[1])
axes[1].set_title("Salary by department")
axes[1].tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()


## 5. Feature engineering and target definition

The target is the observed `Salary` column. `Tenure_year` is calculated from `Join_Date`. We do not generate a synthetic salary target.

In [ ]:
model_df = clean_df.copy()
model_df["Join_Date"] = pd.to_datetime(model_df["Join_Date"], errors="coerce")
model_df["Tenure_year"] = (
    pd.Timestamp.today().normalize() - model_df["Join_Date"]
).dt.days / 365.25

target = "Salary"
features = [
    "Age", "Tenure_year", "Performance_Score",
    "Status", "Department", "Region", "Remote_Work"
]
display(model_df[features + [target]].head())


## 6. Train and evaluate the model

The model script uses a scikit-learn pipeline. Imputation and one-hot encoding are fitted on the training split only, which helps prevent preprocessing leakage.

In [ ]:
from Model import train_and_evaluate

trained_model, metrics = train_and_evaluate(CLEAN_PATH)
print(metrics)


## 7. Project visualization script

Run the standalone script from the repository root to save the full set of charts in `visualizations/`.

In [ ]:
# Run in a terminal from the project root:
# python visualizations/visualize_project.py


## 8. Interpretation and limitations

- R² measures how much target variance is explained on the held-out test set; it can be negative.
- MAE is the average absolute error in the dataset's salary units.
- A good score on this dataset does not prove that the model generalizes to another company or is fair.
- Do not use this model to make real employment or compensation decisions without extensive validation and governance.